<style>
:root {
  color-scheme: dark;
  --spotify-green: #1ed760;
  --spotify-black: #121212;
  --spotify-surface: #181818;
  --spotify-border: #535353;
  --spotify-text: #f5f5f5;
  --spotify-muted: #b3b3b3;
  --bs-body-bg: var(--spotify-black);
  --bs-body-color: var(--spotify-text);
  --bs-border-color: var(--spotify-border);
  --bs-secondary-color: var(--spotify-muted);
}
body { background: var(--spotify-black); color: var(--spotify-text); }
#title-block-header { background: var(--spotify-surface); border-top: .4rem solid var(--spotify-green); padding: clamp(1rem, 5vw, 2.5rem); }
#title-block-header .title { color: var(--spotify-green); font-weight: 750; letter-spacing: -.04em; }
#title-block-header .subtitle { max-width: 42rem; color: var(--spotify-muted); margin-bottom: 0; }
main h2 { border-bottom: 0; border-left: .3rem solid var(--spotify-green); padding-left: .55rem; color: var(--spotify-text); }
main a, #TOC .nav-link.active { color: var(--spotify-green); }
#TOC a { color: var(--spotify-muted); }
main a { text-decoration: underline; text-underline-offset: .15em; }
main a:hover, #TOC a:hover { color: var(--spotify-text); }
main a:focus-visible, #TOC a:focus-visible { outline: 2px solid var(--spotify-green); outline-offset: 3px; }
#TOC .nav-link.active { border-left-color: var(--spotify-green); font-weight: 650; }
#which-genre-labels-qualify .table > :not(caption) > * > * {
  background-color: var(--spotify-surface);
  color: var(--spotify-text);
  border-color: var(--spotify-border);
}
#which-genre-labels-qualify .table > thead > tr > th { background-color: #282828; }
main img { max-width: 100%; height: auto; }
figcaption { color: var(--spotify-muted); line-height: 1.45; }
</style>

Two Spotify tags can describe similarly energetic tracks while landing in very different places on the mood scale. And a single loudness comparison can change by more than a decibel depending on which genres are included. Both stories start with the unit of analysis: a track ID within a genre, or a track ID counted once across the sample.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display
from matplotlib.ticker import MultipleLocator

csv_path = Path("data/dataset.csv")
if not csv_path.is_file():
    csv_path = Path("dataset.csv")
if not csv_path.is_file():
    raise FileNotFoundError("Place the Spotify CSV at data/dataset.csv. See README.md.")

df = pd.read_csv(csv_path)
complete = df.dropna()
by_genre = complete.drop_duplicates(["track_id", "track_genre"])
tracks = complete.drop_duplicates("track_id")
genre_medians = by_genre.groupby("track_genre").agg(
    energy=("energy", "median"), valence=("valence", "median")
)

ink = "#f5f5f5"
green = "#1ed760"
muted = "#b3b3b3"
grid = "#424242"
paper = "#181818"
plt.rcParams.update({
    "axes.facecolor": paper,
    "axes.labelcolor": ink,
    "axes.edgecolor": muted,
    "axes.spines.right": False,
    "axes.spines.top": False,
    "axes.titlecolor": ink,
    "figure.facecolor": paper,
    "font.family": "DejaVu Sans",
    "text.color": ink,
    "xtick.color": ink,
    "ytick.color": ink,
})

## Similar energy, different valence

Energy is Spotify’s estimate of **intensity and activity**. Valence is its estimate of **musical positivity**. Read across the top of the chart: hardstyle and the party label have similar energy, but party scores much higher on valence. Salsa has the highest valence median in this sample at a lower energy level. Each dot is a genre median. [Spotify audio-feature definitions](https://developer.spotify.com/documentation/web-api/reference/get-audio-features).

In [ ]:
#| output: asis

high_energy = tracks.loc[tracks["energy"].ge(0.65)]
dark_share = high_energy["valence"].lt(0.35).mean()
tight_energy = by_genre.loc[
    by_genre["energy"].between(0.85, 0.95)
    & by_genre["track_genre"].isin(["hardstyle", "party"])
]
valence_check = tight_energy.assign(low_valence=tight_energy["valence"].lt(0.35))
checks = valence_check.groupby("track_genre")["low_valence"].agg(["mean", "size"])
hardstyle, party = (checks.loc[label] for label in ("hardstyle", "party"))
soul, pop = (genre_medians.loc[label] for label in ("soul", "pop"))

display(Markdown(
    f"Across {len(high_energy):,} tracks with energy of at least 0.65, "
    f"**{dark_share:.0%}** fall into the shaded low-valence area. The contrast between hardstyle and party also holds "
    "inside a tighter energy band. Among tracks with energy from 0.85 to 0.95, "
    f"**{hardstyle['mean']:.0%}** of the *hardstyle* tracks score below 0.35 on valence "
    f"({hardstyle['size']:,.0f} tracks in the band), compared with "
    f"**{party['mean']:.0%}** for *party* ({party['size']:,.0f} tracks in the band). "
    f"Near the middle of both scales, **Soul** has median valence **{soul.valence:.3f}** "
    f"and energy **{soul.energy:.3f}**. **Pop** has nearly the same median valence "
    f"(**{pop.valence:.3f}**) and sits higher on energy (**{pop.energy:.3f}**)."
))

In [ ]:
#| label: fig-mood-map
#| fig-cap: "Each dot is a genre median. The red box marks valence below 0.35 and energy of at least 0.65. Soul and Pop give reference points near the center."
#| fig-alt: "Scatterplot of median valence and energy. Hardstyle and Black metal sit inside the red high-energy, low-valence box. Soul is near the center, with Pop slightly above it."

fig, ax = plt.subplots(figsize=(8, 4.7), layout="constrained")
ax.fill_between([0, 0.35], 0.65, 1, color="#53262c", linewidth=0)
ax.set_axisbelow(True)
ax.grid(color=grid)
ax.scatter(genre_medians["valence"], genre_medians["energy"],
           s=29, color=muted, alpha=0.51, linewidths=0)

red = "#ff8994"
for genre, name, label_xy, alignment in [
    ("black-metal", "Black metal", (0.045, 0.885), "left"),
    ("hardstyle", "Hardstyle", (0.330, 0.980), "right"),
]:
    valence, energy = genre_medians.loc[genre, ["valence", "energy"]]
    ax.scatter(valence, energy, s=78, color=red, edgecolors=paper, zorder=3)
    ax.annotate(name, (valence, energy), xytext=label_xy, textcoords="data",
                ha=alignment, va="top", fontsize=10, fontweight="bold", color=red, zorder=4)

for genre, name, offset in [
    ("party", "Party", (6, 7)),
    ("salsa", "Salsa", (6, -17)),
    ("ambient", "Ambient", (6, 8)),
    ("soul", "Soul", (8, -13)),
    ("pop", "Pop", (8, 5)),
]:
    valence, energy = genre_medians.loc[genre, ["valence", "energy"]]
    ax.scatter(valence, energy, s=78, color=green, edgecolors=ink, linewidths=0.8, zorder=3)
    ax.annotate(name, (valence, energy), xytext=offset, textcoords="offset points",
                fontsize=10, fontweight="bold", color=green, zorder=4)

ax.set(xlim=(0, 1), ylim=(0, 1.025), xlabel="Median valence", ylabel="Median energy")
ax.set_yticks([0, 0.25, 0.5, 0.75, 1])
ax.xaxis.set_major_locator(MultipleLocator(0.25))
plt.show()

### What the dataset calls happy

The dataset uses **happy** as a genre tag. That bucket contains tracks by Scooter, The Prophet, and Darren Styles. The publisher describes the column as genre and gives the valence score in a separate audio-feature column. The individual scores can also vary widely within the same tag and artist. [Dataset column definitions](https://huggingface.co/datasets/maharshipandya/spotify-tracks-dataset).

In [ ]:
#| output: asis

tag_valence = genre_medians.loc["happy", "valence"]
scooter_ids = ["0RU3sUD0wsMTKSV4gvppOS", "31JhTEAWmmhZIZTm40pQZr"]
examples = by_genre.loc[by_genre["track_genre"].eq("happy")].set_index("track_id")
nessaja, fish = (examples.loc[track_id] for track_id in scooter_ids)

display(Markdown(
    f"The median valence of the *happy* tag is **{tag_valence:.3f}**. Two Scooter tracks "
    f"under that tag illustrate the spread. *{nessaja['track_name']}* scores "
    f"**{nessaja['valence']:.3f}** on valence and *{fish['track_name']}* scores "
    f"**{fish['valence']:.3f}**. Both have energy above 0.90. "
    "Valence is a separate estimate of the sound. Listeners may hear the tracks differently. "
    "[Listen to Nessaja](https://open.spotify.com/track/0RU3sUD0wsMTKSV4gvppOS) · "
    "[Listen to How Much Is The Fish?](https://open.spotify.com/track/31JhTEAWmmhZIZTm40pQZr)"
))

## Why the explicit-track loudness gap shrinks

Across the full sample, explicit tracks are about 2 dB louder on average. The chart compares two pools: all distinct track IDs, and the distinct track IDs with at least one qualifying genre label. In each pool, we average the explicit tracks and non-explicit tracks separately.

In [ ]:
#| output: asis

groups = by_genre.groupby(["track_genre", "explicit"])["loudness"]
counts = groups.size().unstack(fill_value=0)
eligible_labels = counts.index[counts.min(axis=1).ge(30)]
loudness_means = groups.mean().unstack().loc[eligible_labels]
genre_gaps = loudness_means[True] - loudness_means[False]
eligible_ids = by_genre.loc[by_genre["track_genre"].isin(eligible_labels), "track_id"]
in_eligible_label = tracks["track_id"].isin(eligible_ids)
eligible_tracks = tracks.loc[in_eligible_label]
outside = tracks.loc[~in_eligible_label].groupby("explicit")["loudness"].agg(["mean", "size"])

pooled_means = tracks.groupby("explicit")["loudness"].mean()
eligible_means = eligible_tracks.groupby("explicit")["loudness"].mean()
overall_gap = pooled_means[True] - pooled_means[False]
restricted_gap = eligible_means[True] - eligible_means[False]
non_explicit_shift = eligible_means[False] - pooled_means[False]
explicit_shift = eligible_means[True] - pooled_means[True]

display(Markdown(
    f"The gap goes from **{overall_gap:.2f} dB** to **{restricted_gap:.2f} dB**. "
    f"The non-explicit average rises by **{non_explicit_shift:.2f} dB** "
    f"as {outside.loc[False, 'size']:,.0f} non-explicit tracks leave the pool "
    f"(their mean is {outside.loc[False, 'mean']:.2f} dB). "
    f"The explicit average moves by only **{explicit_shift:.2f} dB**, "
    f"with {outside.loc[True, 'size']:,.0f} explicit tracks outside the qualifying labels. "
    "So most of the change comes from which tracks are in the non-explicit group."
))

In [ ]:
#| label: fig-loudness
#| fig-cap: "The distance between points is the explicit-minus-non-explicit gap. Both rows count distinct Spotify track IDs. The colored bands switch at 60% and 85% of the displayed range, measured from the quieter end."
#| fig-alt: "Two rows compare each explicit-status group's mean loudness. The non-explicit point moves toward the explicit point after restricting the sample to eligible labels. The explicit point barely moves. The background changes from green through yellow to red as loudness increases."

fig, ax = plt.subplots(figsize=(8, 3), layout="constrained")
meter_min, meter_max = -9.6, -5.7
meter_span = meter_max - meter_min
band_edges = [meter_min + fraction * meter_span for fraction in (0, 0.60, 0.85, 1)]
band_colors = ("#254b35", "#514519", "#5b2933")
for left, right, color in zip(band_edges[:-1], band_edges[1:], band_colors):
    ax.axvspan(left, right, facecolor=color, linewidth=0, zorder=0)

non_explicit, explicit = green, ink
for row, means in enumerate([pooled_means, eligible_means]):
    quiet, loud = means[False], means[True]
    ax.hlines(row, quiet, loud, color=muted, linewidth=2)
    ax.annotate(f"{loud - quiet:.2f} dB gap", ((quiet + loud) / 2, row),
                xytext=(0, 12), textcoords="offset points", ha="center", fontsize=9)
    for value, color, side in [(quiet, non_explicit, -1), (loud, explicit, 1)]:
        ax.scatter(value, row, s=66, color=color, edgecolors=paper, linewidths=0.8, zorder=3)
        ax.annotate(f"{value:.2f}", (value, row), xytext=(9 * side, 0),
                    textcoords="offset points", ha="left" if side > 0 else "right",
                    va="center", color=color, fontsize=10)

ax.scatter([], [], color=non_explicit, label="Non-explicit")
ax.scatter([], [], color=explicit, label="Explicit")
ax.legend(frameon=False, loc="lower center", bbox_to_anchor=(0.5, 1), ncol=2)
ax.set(yticks=[0, 1], yticklabels=["All tracks", "Qualifying labels only"],
       xlabel="Mean track loudness (dB)", xlim=(meter_min, meter_max), ylim=(1.5, -0.6))
ax.set_axisbelow(True)
ax.grid(axis="x", color=muted, alpha=0.32)
plt.show()

### Which genre labels qualify?

Check each existing Spotify label separately. Count each track ID once within that label. The label qualifies if it has **at least 30 explicit tracks and at least 30 non-explicit tracks**. The cutoff gives both group averages a minimum number of tracks. It is an analysis choice and the result can depend on it.

A track with several genre labels stays in the restricted pool if **any** of its labels qualifies. Every track in that pool contributes once to its group's overall average. For the separate within-label calculation, the same track can contribute to each of its qualifying labels.

In [ ]:
#| output: asis

excluded_labels = counts.loc[~counts.index.isin(eligible_labels)]
too_few_explicit = excluded_labels[True].lt(30).sum()
too_few_non_explicit = excluded_labels[False].lt(30).sum()
zero_explicit = excluded_labels[True].eq(0).sum()
examples = ["sleep", "classical", "ambient", "piano", "techno", "country", "pop", "hip-hop"]

sleep = by_genre.loc[by_genre["track_genre"].eq("sleep")]
sleep_outside = sleep.loc[~sleep["track_id"].isin(eligible_ids)]

table = [
    "| Label | Explicit tracks | Non-explicit tracks | Qualifies? |",
    "| :--- | ---: | ---: | :--- |",
]
for label in examples:
    row = counts.loc[label]
    verdict = "Yes" if label in eligible_labels else "No"
    table.append(f"| {label} | {row[True]:,} | {row[False]:,} | {verdict} |")

display(Markdown(
    f"**{len(eligible_labels)} of {len(counts)} labels qualify.** "
    f"Of the {len(excluded_labels)} excluded labels, "
    f"**{too_few_explicit}** fall below 30 explicit tracks ({zero_explicit} have zero), "
    f"and **{too_few_non_explicit}** fall below 30 non-explicit tracks.\n\n"
    + "\n".join(table)
    + f"\n\nThe {len(sleep_outside):,} sleep tracks outside the restricted pool "
    + f"average **{sleep_outside['loudness'].mean():.1f} dB**, an example of the quieter tracks the cutoff removes."
    + "\n\n**All excluded labels**\n\n"
    + ", ".join(excluded_labels.index)
    + "."
))

### What happens within a label?

In [ ]:
#| output: asis

display(Markdown(
    f"With each label weighted equally, explicit tracks average **{genre_gaps.mean():.2f} dB louder**. "
    f"In **{genre_gaps.lt(0).sum()} of {len(genre_gaps)} labels** the gap goes the other way, "
    f"including emo ({genre_gaps['emo']:+.2f} dB) and hip-hop ({genre_gaps['hip-hop']:+.2f} dB). "
    f"The median label gap is **{genre_gaps.median():+.2f} dB**."
))

In [ ]:
#| label: fig-genre-loudness-gaps
#| fig-cap: "Each point is one qualifying genre label. The gap is explicit mean loudness minus non-explicit mean loudness. Labels can share track IDs."
#| fig-alt: "Ranked dot plot of the explicit loudness gap across 66 labels. Thirteen points lie below zero, including emo and hip-hop. Iranian has the largest positive gap. A dashed line shows the equal-weight average."

ranked_gaps = genre_gaps.sort_values()
ranks = range(1, len(ranked_gaps) + 1)
mean_gap = ranked_gaps.mean()

fig, ax = plt.subplots(figsize=(8, 5.25), layout="constrained")
ax.set_axisbelow(True)
ax.grid(axis="x", color=grid)
ax.axvline(0, color=muted, linewidth=1.2, label="No gap")
ax.axvline(mean_gap, color=ink, linewidth=1.2, linestyle="--",
           label=f"Equal-weight average ({mean_gap:+.2f} dB)")
ax.scatter(ranked_gaps, ranks,
           color=[red if gap < 0 else green for gap in ranked_gaps], s=23, zorder=3)

examples = [
    ("emo", "Emo", (8, 7), "left"),
    ("hip-hop", "Hip-hop", (-8, 0), "right"),
    ("pop", "Pop", (8, 0), "left"),
    ("british", "British", (8, 7), "left"),
    ("iranian", "Iranian", (-8, 0), "right"),
]
for label, name, offset, align in examples:
    gap = ranked_gaps.loc[label]
    rank = ranked_gaps.index.get_loc(label) + 1
    highlight = red if gap < 0 else green
    ax.scatter(gap, rank, color=highlight, edgecolor=ink, linewidth=0.65, s=54, zorder=4)
    ax.annotate(f"{name}  {gap:+.2f}", (gap, rank), xytext=offset,
                textcoords="offset points", ha=align, va="center",
                fontsize=9, color=ink, zorder=5)

ax.set(xlabel="Explicit minus non-explicit mean loudness (dB)",
       ylabel="Genre rank by gap (smallest first)", ylim=(len(ranked_gaps) + 3, -2))
ax.set_yticks([1, 22, 44, len(ranked_gaps)])
ax.set_xlim(ranked_gaps.min() - 1.6, ranked_gaps.max() + 0.65)
ax.legend(frameon=False, loc="lower center", bbox_to_anchor=(0.5, 1), ncol=2)
plt.show()

## Dataset and counting rules

The [Spotify Tracks Dataset on Kaggle](https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset/data) was collected from the Spotify Web API. Each label starts with the same number of rows, so genre frequency reflects how the dataset was assembled. Spotify’s popularity field is a snapshot score. The CSV has no release dates or per-track play counts.

In [ ]:
#| output: asis

label_sizes = df["track_genre"].value_counts()
multi_label_ids = by_genre["track_id"].duplicated(keep=False)
distinct_multi_label_ids = by_genre.loc[multi_label_ids, "track_id"].nunique()
sample_design = (
    f"{label_sizes.iloc[0]:,} rows per label"
    if label_sizes.nunique() == 1
    else f"{label_sizes.min():,} to {label_sizes.max():,} rows per label"
)

display(Markdown(
    f"The CSV has **{len(df):,} rows across {len(label_sizes)} labels** "
    f"({sample_design}). Cleaning drops {len(df) - len(complete):,} for missing values "
    f"and {len(complete) - len(by_genre):,} as duplicates within track-and-label pairs. "
    f"**{distinct_multi_label_ids:,} IDs** appear under multiple labels. A track ID "
    "identifies a Spotify version, so a recording may still appear under different IDs."
))

### Musical key and mode

The overview below keeps the original notebook’s key and mode exploration. It uses one row per Spotify track ID and skips keys outside the documented pitch-class range.

In [ ]:
#| label: fig-key-mode
#| fig-cap: "Counts of distinct Spotify track IDs by estimated key and mode."
#| fig-alt: "Grouped bar chart comparing the major and minor track counts for the twelve pitch classes."

key_names = ["C", "C♯", "D", "D♯", "E", "F", "F♯", "G", "G♯", "A", "A♯", "B"]
key_counts = (
    tracks.loc[tracks["key"].between(0, 11)]
    .groupby(["key", "mode"]).size().unstack(fill_value=0)
    .reindex(index=range(12), columns=[1, 0], fill_value=0)
    .rename(columns={1: "Major", 0: "Minor"})
)
ax = key_counts.plot.bar(figsize=(8, 3.2), color=[green, ink], width=0.75)
ax.set(xticklabels=key_names, xlabel="Estimated key", ylabel="Distinct track IDs")
ax.tick_params(axis="x", rotation=0)
ax.set_axisbelow(True)
ax.grid(axis="y", color=grid)
ax.legend(frameon=False)
ax.figure.tight_layout()
plt.show()